# Task

Design an in-process logging library that an application can call to record events. Each call should fan out to one or more configured destinations.

### Clarifying Questions

Q: What kind of operations does the library need to support?

The library needs to support two main categories of operations:

Logging calls — the application submits a log record at a given severity level (DEBUG, INFO, WARN, ERROR, FATAL), along with a message and optionally structured metadata/context (e.g. key-value pairs, timestamps).

Configuration — at startup, the application sets up one or more destinations, each with its own minimum severity threshold and its own format. Once configured, the library routes each incoming log record to whichever destinations have a threshold at or below the record's severity level.

Q: So, the number of destinations will be same as the number of severity levels? Say 5 destinations for (DEBUG, INFO, WARN, ERROR, FATAL)?

No, the number of destinations is independent of the number of severity levels. You can configure any number of destinations (one, five, ten, etc.), and each destination simply has its own minimum severity threshold that determines which records it accepts.

Q: If a destination has minimum severity of FATAL, only fatal events will be logged?

Yes, that's correct. A destination with a minimum severity of FATAL will only receive records logged at the FATAL level.

Q: What happens if events of other severity are passed to it?

They are simply filtered out — that destination ignores any record whose severity level is below its configured minimum threshold.

Q: What formats do the destination support?

At minimum, the library should support two built-in formats: plain text (a human-readable string) and JSON (structured key-value output). The design should be open to additional formats being added later without modifying existing code.

Q: Do the destinations need to store the events in some data store?

The destinations write records to an output sink — for example, a file on disk, the console/stdout, or a database. The library's job is to get the formatted bytes to that sink; whether the sink itself persists data long-term is a concern of the sink, not the logging library.

Q: Is the sink pass to the library during intialization?

Yes, the sink (e.g. a file handle, stdout, a database connection) is provided to the library at initialization time as part of configuring each destination.

Q: How are the destinations being referenced? Do they need a unique ID or a name?

For v1, destinations don't need a unique ID or name — they are simply a collection configured at startup and referenced internally by the library. There's no requirement for the application to look up or modify a specific destination by name after initialization.

Q: Does the library need to handle concurrent calls?

Yes, concurrent calls must be handled safely. Specifically, when multiple threads log simultaneously to the same destination, the bytes of one record must never interleave with the bytes of another record on that destination.

Q: How about duplicate events? Is there any chance that the caller will send duplicate events that the library needs to figure out?

Duplicate detection is out of scope for this library. The logging service simply accepts every call the application makes and routes it to the appropriate destinations — it has no responsibility for identifying or deduplicating repeated records.

Q: How many callers does the library need to support simultaneously?

The library doesn't impose a fixed limit on the number of concurrent callers — it should handle whatever concurrency the host application produces. The key constraint is simply that writes to any given destination must be safe under concurrent access, as mentioned earlier.

Q: Is there a size limit on each event or limit on the length of the message and metadata?

No size limit on the message or metadata — the library should accept whatever the application passes in.

Q: What are things that are out of scope?

Here's what's explicitly out of scope for v1:
- Hot-reloading config at runtime — destinations and their settings are fixed after initialization.
- Async or buffered writes — all writes are synchronous in v1, though the design should accommodate async later.
- Remote/network destinations — no shipping logs over the network in v1, but the design should be open to it.
- Hierarchical or named loggers — there's no logger hierarchy (e.g. no parent/child logger relationships) and no named logger lookup.
- Duplicate detection — the library does not identify or suppress repeated records.
- Size limits — no enforcement of message or metadata size.

# REQUIREMENTS

### My Answer
1. The logging service logs events sent from applications to one or more destinations.
2. An event contains a severity level (DEBUG, INFO, WARN, ERROR, FATAL), and a message and optionally structured metadata/context (e.g. key-value pairs, timestamps).
3. The log event can be in either format of plain-text or JSON. The service will need to support additional formats in the future without any code modification.
4. Destinations are setup during initialization of the service. The service can support any number of destinations. 
5. Destination has configurations including a minimum severity threshold, format it supports and a sink where the logs will be output to.
6. An event will be written to one or more destinations as long as the event's severity is same or higher than the destination's minimum threshold.
7. The library needs to guarantee thread safety. When multiple threads log simultaneously to the same destination, the bytes of one record must never interleave with the bytes of another record on that destination.
8. There are no limits on the number of concurrent calls.


Out of Scope:
- Hot-reloading config at runtime — destinations and their settings are fixed after initialization.
- Async or buffered writes — all writes are synchronous in v1, though the design should accommodate async later.
- Remote/network destinations — no shipping logs over the network in v1, but the design should be open to it.
- Hierarchical or named loggers — there's no logger hierarchy (e.g. no parent/child logger relationships) and no named logger lookup.
- Duplicate detection — the library does not identify or suppress repeated records.
- Size limits — no enforcement of message or metadata size.

### Feedback
What went well

You captured the primary capabilities really well — logging calls with severity levels, fan-out to multiple destinations, per-destination thresholds, formats, and sinks are all clearly articulated. That's a solid foundation. 

Your out-of-scope section is excellent. You clearly internalized what you learned from the clarifying questions and translated it directly into boundaries, including async writes, network destinations, and hot-reload — all things that will keep your design focused. 
The thread safety requirement is spot-on. Calling out the non-interleaving guarantee specifically is exactly the right level of precision for a concurrency constraint. 

What needs improvement

One area that's missing is error handling at the destination level — specifically, what happens if a sink throws or fails during a write. Should the caller crash? Should other destinations still receive the record? This is worth capturing as a requirement because it directly affects how you'll structure the write loop in your design. 
You mentioned format is per-destination, but it's worth being explicit that format and destination type vary independently — any format can be paired with any sink. Right now requirement 5 reads as if a destination has 'a format it supports,' which could be interpreted as the sink dictating the format rather than the configuration choosing it freely. 

Minor suggestions

It'd be worth noting that each log record should capture a timestamp at the call site, not just rely on whatever the application passes in. This is a small but meaningful design decision that will come up when you define your record structure. 

### Sample Answer